# Omnilingual ASR — Smoke Test (Phase 1, ADR-0003)

**But** : vérifier qu'Omnilingual ASR 300M charge et transcrit 1 audio dioula sans crash, reproductiblement, sur Colab T4.

**Ne PAS faire dans ce notebook** : benchmark vs NeMo (Phase 3), création du provider Python (Phase 2), intégration en chain (Phase 4).

**Critères de sortie ADR-0003 Phase 1** :
- [ ] Notebook exécute complètement sans crash sur Colab T4
- [ ] Omnilingual 300M charge en < 60 s
- [ ] 1 audio dioula transcrit (output non vide)
- [ ] Doc install reproduit par Ruben sur 2ème session Colab

**Si échec** : documenter dans `docs/benchmarks/0002-omnilingual-env-setup.md` puis escalader Plan B (Djelia) si fairseq2 (issue #61) bloque irrémédiablement.

**Référence** : [ADR-0002](../../docs/adr/0002-ajout-provider-omnilingual.md), [ADR-0003](../../docs/adr/0003-plan-ajout-omnilingual.md)

## Cellule 0 — Vérification du runtime Colab

Avant tout, confirmer qu'on est bien sur GPU T4 (Runtime → Change runtime type → T4 GPU).

In [ ]:
import sys
import platform

print(f"Python   : {sys.version}")
print(f"Platform : {platform.platform()}")

# Vérifier la présence GPU NVIDIA
import subprocess
try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True)
    print("\n--- GPU détecté ---")
    print(out)
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    print("\nERREUR : pas de GPU NVIDIA détecté.")
    print("Activer GPU T4 via : Runtime → Change runtime type → T4 GPU")
    raise SystemExit(1)

## Cellule 1 — Installation `omnilingual-asr` + `fairseq2`

**Versions cibles** (à ajuster face à l'[issue #61](https://github.com/facebookresearch/omnilingual-asr/issues/61) — bug fairseq2 connu) :

| Package | Version cible | Note |
|---|---|---|
| omnilingual-asr | `>=0.1.0` (PyPI) | Apache 2.0 |
| fairseq2 | dernière compatible | `pip install fairseq2` |
| torch | déjà fourni par Colab (2.x + CUDA 12) | NE PAS réinstaller |

**Procédure** : si l'install échoue, documenter le message d'erreur exact dans `0002-omnilingual-env-setup.md` puis ajuster les versions ici.

In [ ]:
# Installation Omnilingual ASR + fairseq2
# IMPORTANT : ne PAS réinstaller torch (déjà optimisé sur Colab pour T4)

!pip install --quiet omnilingual-asr
!pip install --quiet fairseq2

# Vérifier les versions installées
import importlib.metadata as md
for pkg in ["omnilingual-asr", "fairseq2", "torch"]:
    try:
        print(f"{pkg:20s} : {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:20s} : NON INSTALLÉ")

## Cellule 2 — Vérification environnement PyTorch + CUDA

Avant de charger le modèle, confirmer que PyTorch voit le GPU.

In [ ]:
import torch

print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"CUDA version   : {torch.version.cuda}")
print(f"GPU count      : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU 0 name     : {torch.cuda.get_device_name(0)}")
    print(f"GPU 0 memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

assert torch.cuda.is_available(), "GPU CUDA requis pour Omnilingual — activer T4 GPU dans Colab"

## Cellule 3 — Import Omnilingual + vérification `dyu_Latn` présent

Confirmer que la langue cible (dioula CI = `dyu_Latn`) figure bien dans la liste des langues supportées.

In [ ]:
# Import du package omnilingual_asr
# NOTE : l'API exacte dépend de la version installée. Ajuster selon docs officielles
# https://github.com/facebookresearch/omnilingual-asr

try:
    import omnilingual_asr
    print(f"omnilingual_asr importé OK — module : {omnilingual_asr.__file__}")
except ImportError as e:
    print(f"ERREUR import : {e}")
    raise

# Vérifier la présence de dyu_Latn dans les langues supportées
# L'attribut exact (lang_ids, supported_languages, etc.) est à confirmer en lisant le code source.
# Tentatives génériques :
try:
    from omnilingual_asr.models.wav2vec2_asr.lang_ids import LANG_IDS
    langs = LANG_IDS
except ImportError:
    try:
        from omnilingual_asr import lang_ids
        langs = lang_ids.LANG_IDS
    except (ImportError, AttributeError):
        langs = None
        print("À AJUSTER : importer la liste des langues depuis le bon chemin (cf. README omnilingual-asr)")

if langs is not None:
    print(f"Total langues supportées : {len(langs)}")
    for code in ["dyu_Latn", "bam_Latn", "bci_Latn", "ann_Latn", "fra_Latn"]:
        present = code in langs
        marker = "✓" if present else "✗"
        print(f"  {marker} {code}")

## Cellule 4 — Chargement du modèle Omnilingual CTC 300M

Mesurer le temps de chargement (cible ADR-0003 : < 60 s) et la mémoire consommée.

In [ ]:
import time
import psutil

process = psutil.Process()
ram_before = process.memory_info().rss / 1e9
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    vram_before = torch.cuda.memory_allocated() / 1e9

print("Téléchargement et chargement du modèle Omnilingual CTC 300M...")
t0 = time.time()

# API exacte à confirmer après lecture du README. Squelette typique :
# from omnilingual_asr import OmnilingualASR
# model = OmnilingualASR.from_pretrained("facebook/omniASR-CTC-300M")
#
# OU via fairseq2 :
# from fairseq2.models.utils import load_model
# model = load_model("omnilingual_asr_ctc_300m")

# TODO Phase 1 : remplir l'API correcte après premier essai d'install réussi
raise NotImplementedError(
    "À COMPLÉTER : API de chargement à valider après install OK. "
    "Documenter les imports corrects dans 0002-omnilingual-env-setup.md."
)

t_load = time.time() - t0
ram_after = process.memory_info().rss / 1e9
if torch.cuda.is_available():
    vram_after = torch.cuda.memory_allocated() / 1e9
    vram_peak = torch.cuda.max_memory_allocated() / 1e9

print(f"\nTemps de chargement : {t_load:.1f} s (cible ADR-0003 : < 60 s)")
print(f"RAM avant : {ram_before:.2f} GB | après : {ram_after:.2f} GB | delta : {ram_after - ram_before:+.2f} GB")
if torch.cuda.is_available():
    print(f"VRAM avant : {vram_before:.2f} GB | après : {vram_after:.2f} GB | peak : {vram_peak:.2f} GB")

## Cellule 5 — Préparation d'un audio de test (Common Voice dyu)

On utilise un clip Common Voice dyu v24 déjà disponible dans le projet (corpus public CC0).

**Option 1** (recommandée) : monter Google Drive et lire un MP3 du dossier `cv-corpus-24.0-2025-12-05-dyu/clips/` que tu as uploadé.

**Option 2** : télécharger un MP3 unique directement depuis HuggingFace Datasets `mozilla-foundation/common_voice_24_0` (subset `dyu`).

In [ ]:
# Option 1 : Google Drive (à privilégier si Ruben a déjà uploadé le corpus)
# from google.colab import drive
# drive.mount("/content/drive")
# audio_path = "/content/drive/MyDrive/wourri/cv-corpus-24.0-2025-12-05-dyu/clips/common_voice_dyu_XXXXXXXX.mp3"

# Option 2 : télécharger 1 sample depuis HuggingFace (sans token, dataset public)
from datasets import load_dataset

print("Chargement d'un échantillon Common Voice dyu v24 (streaming, pas de download massif)...")
ds = load_dataset(
    "mozilla-foundation/common_voice_24_0",
    "dyu",
    split="validation",
    streaming=True,
)
sample = next(iter(ds))

print(f"Référence    : {sample['sentence']}")
print(f"Sample rate  : {sample['audio']['sampling_rate']} Hz")
print(f"Durée audio  : {len(sample['audio']['array']) / sample['audio']['sampling_rate']:.2f} s")

audio_array = sample["audio"]["array"]
sample_rate = sample["audio"]["sampling_rate"]
reference = sample["sentence"]

## Cellule 6 — Transcription du clip dioula

Inférence sur l'audio chargé en cellule 5, langue `dyu_Latn`.

In [ ]:
# API exacte à valider en Phase 1 après lecture du README/exemples officiels

t0 = time.time()

# Squelette (à compléter selon API réelle) :
# transcription = model.transcribe(audio_array, sampling_rate=sample_rate, lang="dyu_Latn")

raise NotImplementedError(
    "À COMPLÉTER : signature exacte de transcribe() à confirmer après cellule 4."
)

t_inf = time.time() - t0
duration_audio = len(audio_array) / sample_rate
rtf = t_inf / duration_audio

print(f"\nRéférence   : {reference}")
print(f"Transcription : {transcription}")
print(f"\nLatence     : {t_inf:.2f} s pour {duration_audio:.2f} s d'audio")
print(f"RTF         : {rtf:.3f} (Real-Time Factor)")

## Cellule 7 — Validation des critères de sortie

Récapitulatif Phase 1 : remplir manuellement après exécution des cellules 1-6.

In [ ]:
# Checklist ADR-0003 Phase 1
criteres = {
    "Notebook exécute sans crash"            : None,  # True / False après run complet
    "Modèle 300M charge en < 60 s"           : None,  # remplir avec t_load mesuré
    "1 audio dioula transcrit (non vide)"    : None,  # True si transcription != ""
    "Reproduit sur 2ème session Colab"       : None,  # à valider lors d'un 2ème run
}

print("=== Critères de sortie Phase 1 ===")
for critere, statut in criteres.items():
    marker = "✓" if statut is True else ("✗" if statut is False else "?")
    print(f"  {marker}  {critere}")

print("\nReporter ces résultats dans docs/benchmarks/0002-omnilingual-env-setup.md.")